In [1]:
#the optimal neural network hyperparameters

In [2]:
import sys
sys.path.insert(1, "C:/Users/hp/Downloads/RL-X/one_policy_to_run_them_all/sentence_transformer")

import optuna

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import csv
from torch.utils.data import Dataset, DataLoader, TensorDataset
from semanticreasoning import ProcessData


c:\ProgramData\anaconda3\envs\robotics\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#load tainset

In [4]:
with open("../commands.csv", "r") as f:
    reader = csv.reader(f)

    next(reader)

    commands = []
    texts = []
    for row in reader:
        if not any(row):
            continue
        text = row[0]
        texts.append(text)
        command = [float(x) for x in row[1:4]]
        commands.append(command)


In [5]:
transformer = ProcessData()

embeddings = transformer.getEmbeddings(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1674.54it/s]


In [6]:
embeddings = torch.tensor(embeddings, dtype=torch.float32)
commands = torch.tensor(commands, dtype=torch.float32)
train_dataset = TensorDataset(embeddings, commands)
train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_6528\3212805297.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embeddings = torch.tensor(embeddings, dtype=torch.float32)


In [7]:
#loading test set

with open("../validation_commands.csv", "r") as f:
    reader = csv.reader(f)

    next(reader)

    commands = []
    texts = []
    for row in reader:
        if not any(row):
            continue
        text = row[0]
        texts.append(text)
        command = [float(x) for x in row[1:4]]
        commands.append(command)


In [8]:
embeddings = transformer.getEmbeddings(texts)

In [9]:
embeddings = torch.tensor(embeddings, dtype=torch.float32)
commands = torch.tensor(commands, dtype=torch.float32)
valid_dataset = TensorDataset(embeddings, commands)
valid_loader = DataLoader(train_dataset, batch_size=20, shuffle=False)

C:\Users\hp\AppData\Local\Temp\ipykernel_6528\689140871.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embeddings = torch.tensor(embeddings, dtype=torch.float32)


In [10]:
OUTPUT = 3
INPUT_SIZE = 384
DEVICE = torch.device("cpu")
EPOCHS = 1000

In [11]:
def define_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []

    in_features = INPUT_SIZE
    for i in range(n_layers):
        out_features = trial.suggest_int("n_units{}".format(i), 4, 128)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float("dropout_l{}".format(i), 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features

    layers.append(nn.Linear(in_features, OUTPUT))

    return nn.Sequential(*layers)

In [12]:
def objective(trial):
    #generate the model
    model = define_model(trial).to(DEVICE)

    #generate the optimizers
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    #train model
    for epoch in range(EPOCHS):
        #training
        model.train()
        for data, target in train_loader:
            
            data, target = data.float().to(DEVICE), target.float().to(DEVICE)

            optimizer.zero_grad()
            predictions = model(data)
            loss = loss_fn(predictions, target)
            loss.backward()
            optimizer.step()

        #validation 
        model.eval()

        total_squared_error = 0.0
        total_elements = 0
        with torch.no_grad():
            for data, target in valid_loader:
                data, target = data.float().to(DEVICE), target.float().to(DEVICE)
                predictions = model(data)

                total_squared_error += (
                    (predictions-target).pow(2).sum().item()
                )

                total_elements += target.numel()

        validation_mse = total_squared_error / total_elements

        trial.report(validation_mse, epoch)

        #Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
    return validation_mse

In [13]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)

study.optimize(objective, n_trials=100)

print("Best validation MSE:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-08-14 09:51:17,879] A new study created in memory with name: no-name-f009ffb7-4dfa-4b01-9947-52e0e966633d
[I 2026-08-14 09:51:43,532] Trial 0 finished with value: 0.07320535065549792 and parameters: {'n_layers': 2, 'n_units0': 128, 'dropout_l0': 0.47632231177307127, 'n_units1': 25, 'dropout_l1': 0.2952022830308421, 'optimizer': 'Adam', 'lr': 2.8028590195256555e-05}. Best is trial 0 with value: 0.07320535065549792.
[I 2026-08-14 09:51:59,695] Trial 1 finished with value: 0.12035859078168869 and parameters: {'n_layers': 1, 'n_units0': 7, 'dropout_l0': 0.4613696104725917, 'optimizer': 'RMSprop', 'lr': 0.0015009278600720494}. Best is trial 0 with value: 0.07320535065549792.
[I 2026-08-14 09:52:27,735] Trial 2 finished with value: 0.08448365384882146 and parameters: {'n_layers': 3, 'n_units0': 123, 'dropout_l0': 0.40717951974973765, 'n_units1': 45, 'dropout_l1': 0.22645386376970672, 'n_units2': 80, 'dropout_l2': 0.3973296509813667, 'optimizer': 'RMSprop', 'lr': 1.903788130198041e-05

Best validation MSE: 0.0008236141169838833
Best parameters: {'n_layers': 2, 'n_units0': 109, 'dropout_l0': 0.28394304287738714, 'n_units1': 46, 'dropout_l1': 0.21347231731331914, 'optimizer': 'Adam', 'lr': 0.0006135036491405206}
